# chip-lakehouse: exploration

Interactive scratch space for poking at the lakehouse - closer to a Databricks-notebook workflow than re-running a whole script to check one thing.

**This notebook is for exploration only.** No pipeline logic lives here - it imports and calls the same `src/` modules the real pipeline stages use (`bronze_ingest.py`, `silver_transform.py`, `gold_marts.py`, `gold_star.py`). See `docs/standard.md` for why the two stay separate.

Requires the Docker Unity Catalog server to be running: `cd docker && docker compose up -d`.

In [1]:
import sys
from pathlib import Path

# notebooks/ is a sibling of src/, not inside it - add src/ to the path so
# the pipeline modules import the same way they do when run as scripts.
sys.path.insert(0, str(Path.cwd().parent / "src"))

from spark_session import get_spark, CATALOG_NAME

spark = get_spark()
spark

:: loading settings :: url = jar:file:/Users/habee1/Desktop/AI-ML/DataPortfolio/fintech-lakehouse/finenv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/habee1/.ivy2/cache
The jars for the packages stored in: /Users/habee1/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
io.unitycatalog#unitycatalog-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5f27de9-8a48-4863-b271-5ef170b6d6a5;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found io.unitycatalog#unitycatalog-spark_2.12;0.2.1 in central
	found io.unitycatalog#unitycatalog-client;0.2.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.23.1 in central
	found org.apache.logging.log4j#log4j-api;2.23.1 in central


	found org.apache.logging.log4j#log4j-core;2.23.1 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jsr310;2.17.0 in central
	found org.openapitools#jackson-databind-nullable;0.2.6 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found com.fasterxml.jackson.core#jackson-databind;2.15.0 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.15.0 in central
	found com.fasterxml.jackson.core#jackson-core;2.15.0 in central
	found com.fasterxml.jackson.module#jackson-module-scala_2.12;2.15.0 in central
	found com.thoughtworks.paranamer#paranamer;2.8 in central
	found com.fasterxml.jackson.dataformat#jackson-dataformat-xml;2.15.0 in central
	found org.codehaus.woodstox#stax2-api;4.2.1 in central
	found com.fasterxml.woodstox#woodstox-core;6.5.1 in central
	found org.antlr#antlr4;4.9.3 in central
	found org.antlr#antlr-runtime;3.5.2 in central
	found org.antlr#ST4;4.3.1 in central
	found org.abego.treelayout#org.abego.treelayout.core;1.0.3 in cen

26/09/06 22:48:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## What's in the catalog

In [2]:
for schema in ["bronze", "silver", "gold", "ml"]:
    print(f"=== {schema} ===")
    spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{schema}").show(truncate=False)

=== bronze ===


+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|bronze   |accounts     |false      |
|bronze   |savings_goals|false      |
|bronze   |transactions |false      |
|bronze   |users        |false      |
+---------+-------------+-----------+

=== silver ===
+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|silver   |account_types    |false      |
|silver   |accounts         |false      |
|silver   |savings_goals    |false      |
|silver   |transaction_types|false      |
|silver   |transactions     |false      |
|silver   |users            |false      |
+---------+-----------------+-----------+

=== gold ===
+---------+----------------------------+-----------+
|namespace|tableName                   |isTemporary|
+---------+----------------------------+-----------+
|gold     |account_summary             |false      |
|gold     |customer_360           

## Gold marts

`customer_360` is built by reading `account_summary` back from the catalog - see `src/gold_marts.py`'s module docstring for why that makes the two marts provably consistent with each other.

In [3]:
spark.table(f"{CATALOG_NAME}.gold.customer_360").orderBy("user_id").limit(10).toPandas()

26/09/06 22:48:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,user_id,age_band,signup_date,num_accounts,total_balance,total_inflows,total_outflows,num_savings_goals,total_target_amount,total_saved_toward_goals,goal_progress_pct
0,00323a26-af72-4349-ba84-c5fe3492d6de,20-29,2025-06-28,3,3159.43,9828.41,6668.98,0,0.00,0.00,None
1,0056fa27-0168-46f9-bdc5-4e89fdc38975,50-59,2025-12-02,2,10552.46,17732.17,7179.71,1,17948.16,14181.04,79.0
2,006e8ae8-a8f3-4763-aad2-4b4d789d7610,70-79,2025-12-17,3,18176.34,30531.11,12354.77,1,9358.20,2468.47,26.4
3,007af90b-6529-4a26-a0a8-1808a757d5ee,70-79,2025-04-09,3,23587.73,40978.50,17390.77,1,15954.65,8862.18,55.5
4,009a985b-0072-4321-abc4-a10abbdc7c16,30-39,2025-01-20,3,8616.99,12223.28,3606.29,0,0.00,0.00,None
5,01278327-767a-4699-a3f5-b0151b567fae,20-29,2025-03-21,2,10078.83,20911.59,10832.76,0,0.00,0.00,None
6,012a6bf4-e28e-4ecf-97f3-90920c876014,40-49,2026-02-20,1,3954.23,4443.49,489.26,1,1031.87,890.11,86.3
7,0145c00f-62ad-4b6e-9f36-5368f5c81ba3,30-39,2024-12-15,2,5993.89,6936.12,942.23,1,17956.00,8767.46,48.8
8,0147da3f-8913-48fa-b530-5858e8f50755,60-69,2026-03-15,2,11933.14,19987.19,8054.05,0,0.00,0.00,None
9,01786986-68fc-453a-8778-5494b3688db4,20-29,2024-07-22,1,11408.56,16604.85,5196.29,1,9995.51,7479.39,74.8


In [4]:
spark.table(f"{CATALOG_NAME}.gold.account_summary").orderBy("account_id").limit(10).toPandas()

,account_id,user_id,account_type,opened_date,total_inflows,total_outflows,balance,transaction_count,first_transaction_ts,last_transaction_ts
0,001426ad-ce8b-49c9-a51e-30e26ba25b48,f58cea3d-a475-40eb-a114-c041ba8bf0bd,pension,2026-02-28,9119.81,8739.84,379.97,48,2026-03-06 03:33:27.723151,2026-08-30 19:38:17.400791
1,001ef9f5-0b25-407e-8c1a-c5f521fc82d0,b4a6d03c-f2b7-4b0e-acbb-8545d4004928,pension,2025-11-12,1156.14,1600.12,-443.98,6,2026-01-01 04:09:44.226895,2026-08-28 14:17:25.198081
2,00241917-4409-4c3e-a522-dbe9cfa10367,3f13ff6d-46cb-4cf6-b543-97320002cea1,pension,2024-04-08,1177.71,855.12,322.59,6,2024-09-30 18:13:21.079171,2026-08-06 11:52:45.684080
3,0036c4ac-163b-4084-a06e-bca654d11471,3ed8714a-5105-4a28-8fab-96be98fb332b,investment,2025-11-08,6309.48,5870.91,438.57,24,2025-11-08 05:30:48.242562,2026-08-07 08:04:43.559868
4,0039c645-c824-481f-bae8-019ff0085216,1b71fda5-dcd8-4738-8add-66faa104d346,savings,2025-11-19,10968.66,5386.06,5582.60,35,2025-11-29 21:05:46.111436,2026-08-30 21:06:34.253110
5,005b53b2-4c6c-43ad-a6df-c6292f244edd,86e30aed-2394-4f0e-945f-57dd130096af,pension,2026-08-25,6610.01,1828.53,4781.48,20,2026-08-25 10:11:09.633526,2026-08-31 12:09:50.595813
6,005d4fdb-de9e-4c80-9b6b-344599e78afb,d8d2a57c-5289-4e95-965d-4bd3cae58af2,investment,2025-06-11,19272.97,6029.44,13243.53,42,2025-06-16 13:14:50.835535,2026-08-18 12:49:27.532431
7,0060c723-416b-4ebd-b5bc-fe153a954091,8ea24f29-2614-4a81-a43c-3227c8d06f70,pension,2024-03-13,11489.20,407.13,11082.07,15,2024-05-09 14:16:55.587821,2026-03-13 15:03:05.252046
8,007010f8-4a1d-4d19-bee7-d221704bf519,eda3ac97-83be-4a0e-b1e8-0cfa7f1ccead,pension,2025-04-25,5147.17,1838.46,3308.71,14,2025-05-01 15:13:58.138536,2026-07-12 14:05:00.991907
9,008df0c0-a27e-43cd-a954-862622869abf,e98e3f79-6204-4e41-a1c7-ebbe7032207c,savings,2024-12-23,2126.03,1711.37,414.66,10,2024-12-23 21:04:37.784814,2026-07-13 12:36:28.091828


## Gold star schema (Kimball)

Conformed dimensional model built by `src/gold_star.py`, in the same `gold` schema as the wide marts above and from the same silver tables. `dim_customer` / `dim_account` are SCD2 (one row per version - `valid_from` / `valid_to` / `is_current`); `dim_date` and `dim_transaction_type` are static overwrites. `fct_transaction` is at transaction grain; `fct_account_monthly_snapshot` is one row per account per month it exists. `tests/verify_e2e.py` reconciles the star back to the marts so the two shapes can't silently disagree.

In [5]:
for t in ["dim_date", "dim_transaction_type", "dim_customer", "dim_account",
          "fct_transaction", "fct_account_monthly_snapshot"]:
    print(f"gold.{t}: {spark.table(f'{CATALOG_NAME}.gold.{t}').count():>8,} rows")

gold.dim_date:    1,127 rows


gold.dim_transaction_type:        4 rows


gold.dim_customer:    2,000 rows


gold.dim_account:    4,020 rows


gold.fct_transaction:  110,604 rows


gold.fct_account_monthly_snapshot:   38,106 rows


In [6]:
spark.table(f"{CATALOG_NAME}.gold.dim_date").orderBy("date_key").limit(10).toPandas()

,date_key,date,year,quarter,month,month_name,day_of_month,day_of_week,day_name,week_of_year,is_weekend,is_month_end
0,20230801,2023-08-01,2023,3,8,August,1,2,Tuesday,31,False,False
1,20230802,2023-08-02,2023,3,8,August,2,3,Wednesday,31,False,False
2,20230803,2023-08-03,2023,3,8,August,3,4,Thursday,31,False,False
3,20230804,2023-08-04,2023,3,8,August,4,5,Friday,31,False,False
4,20230805,2023-08-05,2023,3,8,August,5,6,Saturday,31,True,False
5,20230806,2023-08-06,2023,3,8,August,6,7,Sunday,31,True,False
6,20230807,2023-08-07,2023,3,8,August,7,1,Monday,32,False,False
7,20230808,2023-08-08,2023,3,8,August,8,2,Tuesday,32,False,False
8,20230809,2023-08-09,2023,3,8,August,9,3,Wednesday,32,False,False
9,20230810,2023-08-10,2023,3,8,August,10,4,Thursday,32,False,False


In [7]:
spark.table(f"{CATALOG_NAME}.gold.dim_transaction_type").orderBy("transaction_type_key").toPandas()

,transaction_type_key,type_name,direction,description
0,1,deposit,inflow,Manual deposit into the account
1,2,withdrawal,outflow,Withdrawal out of the account
2,3,roundup,inflow,Spare change swept in from a linked card purchase
3,4,investment_contribution,inflow,Contribution into an investment sub-account


In [8]:
# SCD2: one row per customer version. Filter is_current for the live view.
spark.table(f"{CATALOG_NAME}.gold.dim_customer").orderBy("customer_key").limit(10).toPandas()

,customer_key,user_id,age_band,signup_date,row_hash,valid_from,valid_to,is_current
0,1,00323a26-af72-4349-ba84-c5fe3492d6de,20-29,2025-06-28,563fc29b7a4f4d83832e1020306bd85ed022e21eb382c7...,2025-06-28,9999-12-31,True
1,2,0056fa27-0168-46f9-bdc5-4e89fdc38975,50-59,2025-12-02,ab50a442ede90f9a63920b54ad11ccbf92086de8319217...,2025-12-02,9999-12-31,True
2,3,006e8ae8-a8f3-4763-aad2-4b4d789d7610,70-79,2025-12-17,36ffd73b1f3f39794c5e17994718808e4d0a68fc32e821...,2025-12-17,9999-12-31,True
3,4,007af90b-6529-4a26-a0a8-1808a757d5ee,70-79,2025-04-09,3362b1516421b26c9789c9345c9d1f5f344000a58be92f...,2025-04-09,9999-12-31,True
4,5,009a985b-0072-4321-abc4-a10abbdc7c16,30-39,2025-01-20,836164af939c7108214998fc8ccc13bbedbad09d3691e6...,2025-01-20,9999-12-31,True
5,6,01278327-767a-4699-a3f5-b0151b567fae,20-29,2025-03-21,5acaf1afbe053d6d3b258f4d22f17db2f06f4731fbcbcd...,2025-03-21,9999-12-31,True
6,7,012a6bf4-e28e-4ecf-97f3-90920c876014,40-49,2026-02-20,ec8b2c410c93fb0fd4f76cbc249344e611608844365fff...,2026-02-20,9999-12-31,True
7,8,0145c00f-62ad-4b6e-9f36-5368f5c81ba3,30-39,2024-12-15,cdc6c6f3bb9f2c491c1aff678357b1b56b429ca1727726...,2024-12-15,9999-12-31,True
8,9,0147da3f-8913-48fa-b530-5858e8f50755,60-69,2026-03-15,a7596f6ef5f760639fd8e1c2487b97e3fd620c498432ef...,2026-03-15,9999-12-31,True
9,10,01786986-68fc-453a-8778-5494b3688db4,20-29,2024-07-22,7be3393e649e5ae06890f051de15a0f6ef54b8d6ccd0dc...,2024-07-22,9999-12-31,True


In [9]:
spark.table(f"{CATALOG_NAME}.gold.dim_account").orderBy("account_key").limit(10).toPandas()

,account_key,account_id,user_id,account_number_masked,opened_date,account_type,account_type_category,row_hash,valid_from,valid_to,is_current
0,1,001426ad-ce8b-49c9-a51e-30e26ba25b48,f58cea3d-a475-40eb-a114-c041ba8bf0bd,****3356,2026-02-28,pension,retirement,c7a8746c3d0d25a89076c2f2ca859e73c9a148ece26d2d...,2026-02-28,9999-12-31,True
1,2,001ef9f5-0b25-407e-8c1a-c5f521fc82d0,b4a6d03c-f2b7-4b0e-acbb-8545d4004928,****0676,2025-11-12,pension,retirement,7add56873d74255dd1706e6b24b536a5a4ec23a9cf14e6...,2025-11-12,9999-12-31,True
2,3,00241917-4409-4c3e-a522-dbe9cfa10367,3f13ff6d-46cb-4cf6-b543-97320002cea1,****0726,2024-04-08,pension,retirement,dbb96bc9b16c90d9eaef3aa4c1fae85ad906f3b8ee5667...,2024-04-08,9999-12-31,True
3,4,0036c4ac-163b-4084-a06e-bca654d11471,3ed8714a-5105-4a28-8fab-96be98fb332b,****2384,2025-11-08,investment,investment,984d1c8f954ec3030e8dc6b8d031903c1a14ea80415729...,2025-11-08,9999-12-31,True
4,5,0039c645-c824-481f-bae8-019ff0085216,1b71fda5-dcd8-4738-8add-66faa104d346,****5640,2025-11-19,savings,cash,dd36f02960a16d38b5f439b14d863f0fcd7242a7124b00...,2025-11-19,9999-12-31,True
5,6,005b53b2-4c6c-43ad-a6df-c6292f244edd,86e30aed-2394-4f0e-945f-57dd130096af,****6504,2026-08-25,pension,retirement,0c842944070f7890232454cd9b4f9d65a2fa91007003d5...,2026-08-25,9999-12-31,True
6,7,005d4fdb-de9e-4c80-9b6b-344599e78afb,d8d2a57c-5289-4e95-965d-4bd3cae58af2,****4257,2025-06-11,investment,investment,b2e1ce3de66818352fa827b7d6441cc6b8a40dc192b41a...,2025-06-11,9999-12-31,True
7,8,0060c723-416b-4ebd-b5bc-fe153a954091,8ea24f29-2614-4a81-a43c-3227c8d06f70,****5078,2024-03-13,pension,retirement,de2ee207ec7a1f08a8fcefb3f2cb91fb3ec17f34a784c2...,2024-03-13,9999-12-31,True
8,9,007010f8-4a1d-4d19-bee7-d221704bf519,eda3ac97-83be-4a0e-b1e8-0cfa7f1ccead,****6031,2025-04-25,pension,retirement,5f4fab4d6c8908d8adcc60eab28105231004fe924d6347...,2025-04-25,9999-12-31,True
9,10,008df0c0-a27e-43cd-a954-862622869abf,e98e3f79-6204-4e41-a1c7-ebbe7032207c,****5869,2024-12-23,savings,cash,f637347fc9a14799662157e5a57d75c25ac8993623ce27...,2024-12-23,9999-12-31,True


In [10]:
spark.table(f"{CATALOG_NAME}.gold.fct_transaction").limit(10).toPandas()

,date_key,customer_key,account_key,transaction_type_key,transaction_id,amount,signed_amount,currency
0,20260819,1803,2502,2,ffeb984c-17ab-4d67-bc6b-0b4114af837e,276.92,-276.92,GBP
1,20260809,1781,2214,2,ffe2aabb-34a0-4211-9183-c013e69248b2,496.04,-496.04,GBP
2,20250617,1082,3613,2,ffcaf058-c8f9-485f-86ee-f46c6c2f9711,388.82,-388.82,GBP
3,20250710,1173,259,2,ffc78963-a5e1-422c-a5b9-cd8bb4a13b31,417.33,-417.33,GBP
4,20260106,315,2073,2,ffbd42a0-9a91-4494-8c55-926bc86005b1,363.71,-363.71,GBP
5,20260307,439,3010,2,ffb6f054-8f88-4280-aa80-f89d97ea5561,765.73,-765.73,GBP
6,20240428,799,4003,2,ffa895de-5249-4a82-ad88-bcbad5b3b1be,329.18,-329.18,GBP
7,20260316,1337,1665,2,ffa61690-4d73-41d8-a2c4-ab402e2e871a,93.25,-93.25,GBP
8,20260121,1516,3713,2,ffa38b9f-74c1-48d9-8fb7-8800555b2fe4,836.52,-836.52,GBP
9,20260730,329,1176,2,ff764cf1-d890-414a-b3b5-e1555f2ee36c,330.23,-330.23,GBP


In [11]:
spark.table(f"{CATALOG_NAME}.gold.fct_account_monthly_snapshot") \
    .orderBy("account_key", "date_key").limit(12).toPandas()

,date_key,customer_key,account_key,month_inflow,month_outflow,month_net,closing_balance,transaction_count
0,20260228,1918,1,0.00,0.00,0.00,0.00,0
1,20260331,1918,1,1657.06,2181.35,-524.29,-524.29,11
2,20260430,1918,1,564.10,2694.19,-2130.09,-2654.38,7
3,20260531,1918,1,381.19,1413.43,-1032.24,-3686.62,7
4,20260630,1918,1,2805.69,1993.31,812.38,-2874.24,10
5,20260731,1918,1,1817.85,457.56,1360.29,-1513.95,5
6,20260831,1918,1,1893.92,0.00,1893.92,379.97,8
7,20251130,1440,2,0.00,0.00,0.00,0.00,0
8,20251231,1440,2,0.00,0.00,0.00,0.00,0
9,20260131,1440,2,482.87,0.00,482.87,482.87,1


## Scratch

Anything ad hoc goes below - a quick `spark.sql(...)`, a `.toPandas()` for a plot, checking a hunch before it becomes real code in `src/`.